# 3. Decoder

Notebook ini mengelola dua belas variasi decoder RNN/LSTM. Default-nya memakai artefak final di `models/rnn` dan tidak melakukan retraining.


In [ ]:
from pathlib import Path
import sys

def find_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start] + list(start.parents):
        if (path / 'src').exists() and (path / 'data').exists():
            return path
    raise RuntimeError('Repo root tidak ditemukan. Jalankan dari repo, Colab, atau Kaggle working directory yang sudah berisi repo.')

ROOT = find_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('ROOT:', ROOT)


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from rnn.sequences import teacher_pairs
from rnn.keras_models import build_preinject, compile_model
from rnn.train import grid_cfg
from rnn.weights import export_weights

SEED = 42
BATCH_SIZE = 128
EPOCHS = 10
EMBED_DIM = 256
LEARNING_RATE = 1e-3
SKIP_RNN_TRAINING = True

PROC_DIR = ROOT / 'data/processed/flickr8k'
MODEL_DIR = ROOT / 'models/rnn'
TABLE_DIR = ROOT / 'reports/tables/rnn'
for folder in [MODEL_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

tf.keras.utils.set_random_seed(SEED)
print('SKIP_RNN_TRAINING:', SKIP_RNN_TRAINING)


In [ ]:
def repo_relative(path):
    path = Path(path)
    try:
        return path.resolve().relative_to(ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()

def artifact_path(value):
    path = Path(str(value))
    return path if path.is_absolute() else ROOT / path

def load_existing_records():
    path = TABLE_DIR / 'train_records.json'
    if not path.exists():
        return []
    records = json.loads(path.read_text(encoding='utf-8'))
    for record in records:
        record['model_path'] = repo_relative(artifact_path(record['model_path']))
        record['weight_path'] = repo_relative(artifact_path(record['weight_path']))
        record['history_path'] = repo_relative(artifact_path(record['history_path']))
    return records

def decoder_artifacts_complete(records):
    if len(records) != 12:
        return False
    for record in records:
        if not artifact_path(record['model_path']).exists():
            return False
        if not artifact_path(record['weight_path']).exists():
            return False
        if not artifact_path(record['history_path']).exists():
            return False
    return True

records = load_existing_records()
print('existing records:', len(records), 'complete:', decoder_artifacts_complete(records))
pd.DataFrame(records).head()


In [ ]:
if SKIP_RNN_TRAINING:
    if not decoder_artifacts_complete(records):
        missing = []
        for record in records:
            for key in ['model_path', 'weight_path', 'history_path']:
                if not artifact_path(record[key]).exists():
                    missing.append(record[key])
        raise RuntimeError('SKIP_RNN_TRAINING=True tetapi artefak decoder belum lengkap. Missing: ' + ', '.join(missing[:10]))
    print('Semua 12 decoder sudah tersedia. Training dilewati.')
else:
    train_features = np.load(PROC_DIR / 'train_features.npy').astype('float32')
    train_captions = np.load(PROC_DIR / 'train_captions.npy').astype('int32')
    val_features = np.load(PROC_DIR / 'val_features.npy').astype('float32')
    val_captions = np.load(PROC_DIR / 'val_captions.npy').astype('int32')
    word_to_index = json.loads((PROC_DIR / 'vocab.json').read_text(encoding='utf-8'))

    def make_dataset(features, captions, shuffle=False):
        inputs, targets = teacher_pairs(captions)
        ds = tf.data.Dataset.from_tensor_slices(((features, inputs), targets))
        if shuffle:
            ds = ds.shuffle(min(len(captions), 10000), seed=SEED, reshuffle_each_iteration=True)
        return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    def history_dict(history):
        return {key: [float(value) for value in values] for key, values in history.history.items()}

    train_ds = make_dataset(train_features, train_captions, shuffle=True)
    val_ds = make_dataset(val_features, val_captions, shuffle=False)
    base_config = {
        'vocab_size': len(word_to_index),
        'feature_dim': int(train_features.shape[1]),
        'max_length': int(train_captions.shape[1]),
        'caption_length': int(train_captions.shape[1] - 1),
        'embed_dim': EMBED_DIM,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
    }

    records = []
    for cfg in grid_cfg(base_config):
        stem = Path(cfg['model_name']).stem
        model_path = MODEL_DIR / cfg['model_name']
        weight_path = MODEL_DIR / f'{stem}.npz'
        history_path = TABLE_DIR / f'{stem}_history.json'
        record = {
            'config': cfg,
            'model_path': repo_relative(model_path),
            'weight_path': repo_relative(weight_path),
            'history_path': repo_relative(history_path),
        }
        if model_path.exists() and weight_path.exists() and history_path.exists():
            print('skip decoder', cfg['model_name'])
            record['skipped_existing'] = True
            records.append(record)
            continue
        print('train decoder', cfg['model_name'])
        tf.keras.backend.clear_session()
        model = build_preinject(
            vocab_size=cfg['vocab_size'], feature_dim=cfg['feature_dim'], max_length=cfg['caption_length'],
            embed_dim=cfg['embed_dim'], hidden_size=cfg['hidden_size'], recur_layers=cfg['recur_layers'], recur_type=cfg['recur_type']
        )
        model = compile_model(model, learn_rate=cfg['learning_rate'])
        history = model.fit(train_ds, validation_data=val_ds, epochs=cfg['epochs'], verbose=1)
        model.save(model_path)
        export_weights(model, weight_path)
        history_path.write_text(json.dumps(history_dict(history), indent=2), encoding='utf-8')
        record['skipped_existing'] = False
        records.append(record)
        (TABLE_DIR / 'train_records.json').write_text(json.dumps(records, indent=2), encoding='utf-8')

    (TABLE_DIR / 'train_records.json').write_text(json.dumps(records, indent=2), encoding='utf-8')

pd.DataFrame(records)
